[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C35_Speech_Audio_Course/03_codecs/03_codecs.ipynb)

# 03 · 神经音频编解码（用 numpy 从零实现）

目标：把 **矢量量化 VQ、残差矢量量化 RVQ、码本利用率、比特率** 用 numpy 从零实现，
并对拍核心不变量——**RVQ 重建误差随级数单调下降**、各级残差范数递减。

路线：VQ 最近邻 → 量化误差 vs K → RVQ 多级 → 重建误差单调(对拍) → 码本利用率/困惑度 → 比特率 → ✏️ 练习 → 📖 答案 → 🧪 EnCodec 胶囊。

> 心智模型：**把向量四舍五入到码本里最近的词，用词的编号代替向量。** RVQ = 反复对残差四舍五入。

## 1 · 矢量量化 VQ：最近邻替换

`k*(x) = argmin_k ||x - c_k||²`，量化值 `q(x) = c_{k*}`。

向量化实现：一次算出所有输入到所有码字的距离矩阵，取每行最小。对拍：量化值确实是码本中离 x 最近的。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def vq_encode(x, codebook):
    '''x:(N,d), codebook:(K,d) -> (下标 (N,), 量化值 (N,d))。'''
    # ||x-c||² = ||x||² - 2 x·c + ||c||²；这里直接算成对距离更直观
    d2 = ((x[:, None, :] - codebook[None, :, :])**2).sum(axis=-1)   # (N, K)
    idx = np.argmin(d2, axis=1)
    return idx, codebook[idx]

d, K, N = 4, 8, 50
codebook = rng.standard_normal((K, d))
x = rng.standard_normal((N, d))
idx, q = vq_encode(x, codebook)
print('下标范围', idx.min(), '..', idx.max(), '| 量化值形状', q.shape)
# 对拍：逐个样本暴力找最近码字
for i in range(N):
    dists = ((codebook - x[i])**2).sum(axis=1)
    assert idx[i] == np.argmin(dists), i
    assert np.allclose(q[i], codebook[idx[i]])
# 量化值就是码本中的某一行
assert all(np.any(np.all(codebook == q[i], axis=1)) for i in range(N))
print('✅ VQ 最近邻正确：每个 x 被替换成码本中欧氏最近的码字')

## 2 · 量化误差随码本增大而下降

率失真的最朴素形态：码本越大 `K`，平均量化误差越小，但每 token 比特 `log2(K)` 越大。
我们用同一份数据、k-means 风格的码本，验证误差随 K **单调下降**。

In [ ]:
def kmeans_codebook(x, K, iters=20, seed=0):
    '''简单 k-means 学码本（Lloyd 迭代）。'''
    g = np.random.default_rng(seed)
    cb = x[g.choice(len(x), K, replace=False)].copy()
    for _ in range(iters):
        idx, _ = vq_encode(x, cb)
        for k in range(K):
            mask = idx == k
            if mask.any():
                cb[k] = x[mask].mean(axis=0)
    return cb

def mse(x, codebook):
    _, q = vq_encode(x, codebook)
    return float(((x - q)**2).mean())

x = rng.standard_normal((400, 4))
errs = []
for K in [2, 4, 8, 16, 32, 64]:
    cb = kmeans_codebook(x, K, seed=1)
    e = mse(x, cb)
    errs.append(e)
    print(f'K={K:3d}  ({np.log2(K):.0f} bit/token)  量化 MSE = {e:.4f}')
# 单调下降（k-means 局部最优偶有微小波动，留 5% 容差）
for i in range(1, len(errs)):
    assert errs[i] <= errs[i-1]*1.05 + 1e-9, (i, errs)
assert errs[-1] < errs[0]
print('✅ 码本越大，量化误差越小（代价：每 token 比特数越多）—— 率失真权衡')

## 3 · 残差矢量量化 RVQ：逐级量化残差

`r0=x; 级i: 量化 r_{i-1} -> c_i, r_i = r_{i-1} - c_i; 重建 = Σ c_i`。

用 N 个小码本逐级逼近。返回每级下标与最终重建。

In [ ]:
def rvq_encode(x, codebooks):
    '''codebooks: list of (K,d)。返回 (各级下标 list, 重建 (N,d), 各级残差范数)。'''
    residual = x.copy()
    idxs, recon = [], np.zeros_like(x)
    res_norms = [float(np.linalg.norm(residual))]
    for cb in codebooks:
        i, q = vq_encode(residual, cb)
        idxs.append(i)
        recon = recon + q
        residual = residual - q
        res_norms.append(float(np.linalg.norm(residual)))
    return idxs, recon, res_norms

def rvq_decode(idxs, codebooks):
    '''查表求和还原。'''
    recon = None
    for i, cb in zip(idxs, codebooks):
        q = cb[i]
        recon = q if recon is None else recon + q
    return recon

d, K, N = 4, 16, 100
x = rng.standard_normal((N, d))
# 每级码本：对当前残差做 k-means（真实系统是联合训练，这里逐级拟合足以演示机制）
codebooks, residual = [], x.copy()
for _ in range(4):
    cb = kmeans_codebook(residual, K, seed=2)
    codebooks.append(cb)
    _, q = vq_encode(residual, cb)
    residual = residual - q
idxs, recon, res_norms = rvq_encode(x, codebooks)
print('RVQ 级数', len(idxs), '| 每级下标形状', idxs[0].shape)
# 编码-解码一致
assert np.allclose(recon, rvq_decode(idxs, codebooks))
print('各级后残差范数:', [round(r, 2) for r in res_norms])
print('✅ RVQ 编码/解码一致；残差范数逐级下降')

## 4 · 核心不变量：重建误差随级数单调下降 ★

RVQ 的灵魂：**用前 n 级重建，n 越大误差越小**（每级都在缩小残差）。
这同时也是『可变比特率』的来源——只用前 n 级 = 更低比特率 + 略差质量，无需重训。

In [ ]:
def rvq_partial_recon(idxs, codebooks, n_levels):
    '''只用前 n_levels 级重建。'''
    return rvq_decode(idxs[:n_levels], codebooks[:n_levels])

errs = []
for n in range(1, len(codebooks)+1):
    r = rvq_partial_recon(idxs, codebooks, n)
    e = float(((x - r)**2).mean())
    errs.append(e)
    print(f'用前 {n} 级重建 -> MSE = {e:.4f}  (比特率 ∝ {n})')
# ★ 严格单调下降
for i in range(1, len(errs)):
    assert errs[i] < errs[i-1] + 1e-12, (i, errs)
# 残差范数也应单调下降
assert all(res_norms[i+1] <= res_norms[i] + 1e-9 for i in range(len(res_norms)-1))
print('✅ ★ 重建误差随 RVQ 级数严格单调下降 —— 这就是可变比特率的来源')

## 5 · 码本利用率与困惑度：坍缩的度量

**码本利用率** = 被用到的码字 / 码本大小。**困惑度** = exp(下标分布的熵) = 有效码字数。
码本远大于数据多样性时，利用率会暴跌（坍缩）。我们构造对比来量化它。

In [ ]:
def codebook_stats(idx, K):
    '''返回 (利用率, 困惑度)。'''
    counts = np.bincount(idx, minlength=K).astype(float)
    used = (counts > 0).sum()
    utilization = used / K
    probs = counts / counts.sum()
    nz = probs[probs > 0]
    entropy = -(nz * np.log(nz)).sum()
    perplexity = float(np.exp(entropy))    # 有效码字数
    return float(utilization), perplexity

# 情形A：数据多样性 >> 码本 -> 高利用率
x_diverse = rng.standard_normal((2000, 4))
cbA = kmeans_codebook(x_diverse, 16, seed=3)
idxA, _ = vq_encode(x_diverse, cbA)
uA, pA = codebook_stats(idxA, 16)
# 情形B：数据其实只有 3 个簇，却给 64 大码本 -> 坍缩
centers = rng.standard_normal((3, 4)) * 5
x_clustered = centers[rng.integers(0, 3, 2000)] + 0.1*rng.standard_normal((2000, 4))
cbB = rng.standard_normal((64, 4))             # 随机大码本(未训练好)
idxB, _ = vq_encode(x_clustered, cbB)
uB, pB = codebook_stats(idxB, 64)
print(f'A 多样数据/16码本 : 利用率={uA:.0%}  困惑度={pA:.1f}')
print(f'B 3簇数据/64码本  : 利用率={uB:.0%}  困惑度={pB:.1f} (有效码字远<64 -> 坍缩)')
assert uA > uB, '数据多样且码本合适时利用率应更高'
assert pB < 64*0.5, '坍缩时有效码字数远小于码本大小'
print('✅ 码本利用率/困惑度量化了坍缩：码本远大于数据多样性 -> 浪费比特')

## 6 · 比特率：折算成每秒多少比特

`bitrate = 帧率 × 级数 N × log2(K)`（bps）。
对拍：用真实 EnCodec 配置算出的比特率，应等于按定义逐项相乘。

In [ ]:
def bitrate(frame_rate, n_levels, codebook_size):
    '''返回 bps。'''
    bits_per_level = np.log2(codebook_size)
    return frame_rate * n_levels * bits_per_level

# EnCodec 24kHz 的一档真实配置：帧率~75, N=8, K=1024
br = bitrate(frame_rate=75, n_levels=8, codebook_size=1024)
print(f'EnCodec(75帧/s, 8级, K=1024) -> {br:.0f} bps = {br/1000:.1f} kbps')
assert br == 75*8*10, '75*8*log2(1024)=75*8*10'
# 原始 PCM 比特率 vs 压缩比
pcm = 24000 * 16          # 24kHz * 16bit
print(f'原始 PCM = {pcm/1000:.0f} kbps；压缩比 = {pcm/br:.0f}x')
assert pcm/br > 60
# 只用前 n 级 -> 可变比特率
for n in [1, 2, 4, 8]:
    print(f'  只用前 {n} 级 -> {bitrate(75, n, 1024)/1000:.1f} kbps')
print('✅ 比特率 = 帧率×级数×log2(K)；减级数即降比特率（同一模型多档）')

---
## ✏️ 练习 1：VQ 最近邻量化

实现 `quantize(x, codebook)`：返回 `(下标, 量化值)`。要求向量化（用成对距离），不要 python 双循环。

In [ ]:
def quantize(x, codebook):
    # TODO: 算 (N,K) 距离矩阵 d2[n,k]=||x[n]-codebook[k]||²; idx=argmin; 返回 idx, codebook[idx]
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
cb = rng.standard_normal((10, 3))
x = rng.standard_normal((20, 3))
idx, q = quantize(x, cb)
assert idx.shape == (20,) and q.shape == (20, 3)
for i in range(20):
    assert idx[i] == np.argmin(((cb - x[i])**2).sum(1))
print('✅ 练习 1 通过：VQ 最近邻量化正确')

## ✏️ 练习 2：RVQ 残差迭代

实现 `residual_quantize(x, codebooks)`：逐级量化残差，返回 `(各级下标 list, 重建)`。

In [ ]:
def residual_quantize(x, codebooks):
    # TODO: residual=x.copy(); recon=0; 对每个 cb: 量化 residual -> q, 记下标,
    #        recon+=q, residual-=q; 返回 (下标list, recon)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
cbs = [rng.standard_normal((8, 3)) for _ in range(3)]
x = rng.standard_normal((30, 3))
idxs, recon = residual_quantize(x, cbs)
assert len(idxs) == 3 and recon.shape == (30, 3)
# 重建应等于各级码字之和
manual = sum(cbs[i][idxs[i]] for i in range(3))
assert np.allclose(recon, manual)
# 多用一级，误差不增
_, r2 = residual_quantize(x, cbs[:2])
assert ((x-recon)**2).mean() <= ((x-r2)**2).mean() + 1e-12
print('✅ 练习 2 通过：RVQ 残差迭代正确，多一级误差不增')

## ✏️ 练习 3：码本利用率

实现 `utilization(idx, K)`：返回被用到的码字比例（0~1）。

In [ ]:
def utilization(idx, K):
    # TODO: 统计 idx 中出现过的不同值个数 / K
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert abs(utilization(np.array([0,0,1,1,2]), 4) - 3/4) < 1e-9   # 用了0,1,2 共3个
assert abs(utilization(np.array([5,5,5]), 8) - 1/8) < 1e-9       # 只用1个
assert abs(utilization(np.arange(16), 16) - 1.0) < 1e-9          # 全用满
print('✅ 练习 3 通过：码本利用率计算正确')

## ✏️ 练习 4：比特率与可变比特率

实现 `compute_bitrate(sample_rate, hop, n_levels, K)`：先算帧率 `sample_rate/hop`，再 `× n_levels × log2(K)`。
用它验证：减半级数 -> 比特率减半。

In [ ]:
def compute_bitrate(sample_rate, hop, n_levels, K):
    # TODO: frame_rate=sample_rate/hop; 返回 frame_rate*n_levels*log2(K)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
# 24kHz, hop=320 -> 75 帧/s; 8级; K=1024 -> 6000 bps
br8 = compute_bitrate(24000, 320, 8, 1024)
assert abs(br8 - 6000) < 1e-6, br8
br4 = compute_bitrate(24000, 320, 4, 1024)
assert abs(br4 - br8/2) < 1e-6, '级数减半 -> 比特率减半'
print(f'8级={br8:.0f} bps, 4级={br4:.0f} bps')
print('✅ 练习 4 通过：比特率公式正确，减级数即降比特率')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def quantize(x, codebook):
    d2 = ((x[:, None, :] - codebook[None, :, :])**2).sum(axis=-1)
    idx = np.argmin(d2, axis=1)
    return idx, codebook[idx]

In [ ]:
# 练习 2 参考答案
def residual_quantize(x, codebooks):
    residual = x.copy()
    idxs, recon = [], np.zeros_like(x)
    for cb in codebooks:
        i, q = quantize(residual, cb)
        idxs.append(i); recon = recon + q; residual = residual - q
    return idxs, recon

In [ ]:
# 练习 3 参考答案
def utilization(idx, K):
    return len(np.unique(idx)) / K

In [ ]:
# 练习 4 参考答案
def compute_bitrate(sample_rate, hop, n_levels, K):
    frame_rate = sample_rate / hop
    return frame_rate * n_levels * np.log2(K)

---
## 🧪 真实数据胶囊：复算 EnCodec / SoundStream 的比特率档位

EnCodec（24 kHz）与 SoundStream 都用 RVQ + 可变比特率。我们用它们**公开的真实配置**复算各比特率档，
验证『同一个模型靠改 RVQ 级数给出多档比特率』。

（纯 numpy/标准库，无需联网、无需 encodec 包。）

In [ ]:
# EnCodec 24kHz 真实配置（公开）：帧率 75 Hz, 码本 K=1024(10bit), 最多 N=32 级
ENCODEC = dict(frame_rate=75, codebook_size=1024)

def encodec_bitrate(n_levels, cfg=ENCODEC):
    return cfg['frame_rate'] * n_levels * np.log2(cfg['codebook_size'])

# EnCodec 论文报告的几档目标比特率 与 对应 RVQ 级数
print(f"{'级数 N':>6} {'比特率':>12}")
targets = {}
for n in [2, 4, 8, 16]:
    br = encodec_bitrate(n)
    targets[n] = br
    print(f'{n:>6} {br/1000:>9.1f} kbps')
# 验证：n=2 -> 1.5 kbps, n=8 -> 6 kbps（EnCodec 的标称档）
assert abs(targets[2] - 1500) < 1e-6
assert abs(targets[8] - 6000) < 1e-6
assert targets[16] == 2*targets[8]
print('\n✅ 复现 EnCodec 比特率档：1.5 / 3 / 6 / 12 kbps 对应 RVQ 2/4/8/16 级')

**🧪 胶囊练习**：实现 `tokens_per_second(frame_rate, n_levels)` 与 `seq_len(audio_sec, frame_rate, n_levels)`：
算出每秒产生多少个离散 token、以及 `audio_sec` 秒音频给语音 LM 的 token 序列总长。
（这直接关系到模块 05 语音 LM 的建模难度——序列越长越难。）

In [ ]:
def tokens_per_second(frame_rate, n_levels):
    # TODO: 每帧 n_levels 个 token -> frame_rate * n_levels
    raise NotImplementedError

def seq_len(audio_sec, frame_rate, n_levels):
    # TODO: audio_sec * tokens_per_second(...)，取 int
    raise NotImplementedError

In [ ]:
# 自测
tps = tokens_per_second(75, 8)
assert tps == 600, '75帧*8级 = 600 token/秒'
L = seq_len(10, 75, 8)
assert L == 6000
# 低帧率(Mimi 风格 12.5Hz)大幅缩短序列
L_low = seq_len(10, 12.5, 8)
print(f'EnCodec 75Hz/8级: {tps} token/s, 10秒={L} tokens')
print(f'低帧率 12.5Hz/8级: 10秒={L_low} tokens (序列短 6 倍, 更利于语音 LM)')
assert L_low < L
print('✅ 胶囊练习通过：会算 token 率与序列长，理解它对语音 LM 的影响')

In [ ]:
# 📖 胶囊参考答案
def tokens_per_second(frame_rate, n_levels):
    return frame_rate * n_levels

def seq_len(audio_sec, frame_rate, n_levels):
    return int(audio_sec * tokens_per_second(frame_rate, n_levels))

### 小结
- **VQ** = 把向量换成码本里最近的码字，用整数下标代替；压成 log2(K) 比特。
- **RVQ** = 逐级量化残差，N 个小码本叠出 K^N 精度；重建误差**随级数单调下降**。
- **可变比特率** = 只用前 n 级，无需重训；这是 RVQ 的杀手特性。
- **码本利用率/困惑度** 度量坍缩；码本远大于数据多样性 -> 浪费比特。
- **比特率** = 帧率×级数×log2(K)；token 序列长度直接影响下游语音 LM。

下一站：**模块 04 · TTS 与声码器** —— 反过来，把谱图还原成能听的波形。